# PostHog GitHub data — exploratory analysis

SQL exploration of the cached `PostHog/posthog` extract loaded into local Postgres.

**Prerequisites:** `docker compose up -d`, `uv sync`, `uv run python scripts/load_db.py`, kernel **Python (weave)**.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "scripts" / "pg.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.pg import read_df


## Database overview

In [2]:
tables = read_df('''
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_type, table_name
''')
display(tables)

display(read_df('''
    SELECT 'prs' AS table_name, COUNT(*) AS rows FROM prs
    UNION ALL SELECT 'issues', COUNT(*) FROM issues
    UNION ALL SELECT 'users', COUNT(*) FROM users
    UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
    UNION ALL SELECT 'pr_labels', COUNT(*) FROM pr_labels
    ORDER BY table_name
'''))


,table_name,table_type
0,fetch_meta,BASE TABLE
1,issue_labels,BASE TABLE
2,issues,BASE TABLE
3,org_members,BASE TABLE
4,pr_labels,BASE TABLE
5,prs,BASE TABLE
6,reviews,BASE TABLE
7,users,BASE TABLE
8,pr_volume,VIEW
9,review_reach,VIEW


,table_name,rows
0,issues,969
1,pr_labels,7578
2,prs,13476
3,reviews,11454
4,users,202


## Pull requests

In [3]:
prs_df = read_df('''
    SELECT number, title, state, comments, author_login, author_type, closed_at
    FROM prs
''')
display(prs_df.head())
prs_df.comments.describe()


,number,title,state,comments,author_login,author_type,closed_at
0,59792,feat(web-analytics): add lazy precompute rollo...,closed,3,lricoy,User,2026-05-23 21:35:59+00:00
1,59787,fix(insights): allow viewed/timing/metalytics ...,closed,4,posthog[bot],Bot,2026-05-23 18:00:38+00:00
2,59784,fix(llma): route personal_spend through events...,closed,5,pauldambra,User,2026-05-23 18:31:54+00:00
3,59781,chore(insights): move retention bar/line chart...,closed,4,sampennington,User,2026-05-23 15:39:13+00:00
4,59779,test: fix reverseProxyCheckerLogic read-only t...,closed,4,arnohillen,User,2026-05-23 12:47:03+00:00


count    13476.000000
mean         5.232191
std          3.081051
min          0.000000
25%          3.000000
50%          5.000000
75%          6.000000
max         96.000000
Name: comments, dtype: float64

In [4]:
display(prs_df.author_type.value_counts().to_frame("count"))

display(
    prs_df.loc[prs_df.author_type != "Bot", "author_login"]
    .value_counts()
    .head(15)
    .to_frame("merged_prs")
)


,count
author_type,
User,12533
Bot,943


,merged_prs
author_login,
Gilbert09,2214
pauldambra,517
andrewm4894,356
sampennington,344
rnegron,286
webjunkie,237
skoob13,217
MattPua,212
jonmcwest,209


## Issues and users

In [5]:
issues_df = read_df('''
    SELECT number, title, author_login, closed_at, comments
    FROM issues
''')
users_df = read_df(
    "SELECT login, name, company, merged_pr_count, is_affiliated FROM users"
)

display(issues_df.head())
display(users_df.head())


,number,title,author_login,closed_at,comments
0,59707,Bug report :`$ai_evaluation` Custom analytics ...,alauxui,2026-05-22 18:14:22+00:00,0
1,59492,test,posthog[bot],2026-05-21 18:36:03+00:00,0
2,59344,Allow custom day labeling in retention cohort ...,hait0ng,2026-05-22 09:08:33+00:00,1
3,59036,ProseMirror-selectednode CSS class is added to...,crash-and-burn,2026-05-21 14:34:38+00:00,1
4,57019,Account tagging and segmentation,arthurdedeus,2026-05-21 23:37:12+00:00,0


,login,name,company,merged_pr_count,is_affiliated
0,Gilbert09,Tom Owers,NaN,2214,True
1,pauldambra,Paul D'Ambra,@posthog,517,True
2,andrewm4894,Andrew Maguire,PostHog,356,True
3,sampennington,Sam Pennington,VEED,344,False
4,rnegron,Raúl Negrón-Otero,@PostHog,286,True


In [6]:
top_authors = (
    prs_df.loc[prs_df.author_type != "Bot", "author_login"]
    .value_counts()
    .head(20)
    .rename_axis("author_login")
    .reset_index(name="merged_prs")
    .merge(
        users_df[["login", "name", "company"]],
        left_on="author_login",
        right_on="login",
        how="left",
    )
)
display(top_authors)


,author_login,merged_prs,login,name,company
0,Gilbert09,2214,Gilbert09,Tom Owers,NaN
1,pauldambra,517,pauldambra,Paul D'Ambra,@posthog
2,andrewm4894,356,andrewm4894,Andrew Maguire,PostHog
3,sampennington,344,sampennington,Sam Pennington,VEED
4,rnegron,286,rnegron,Raúl Negrón-Otero,@PostHog
5,webjunkie,237,webjunkie,Julian Bez,@PostHog
6,skoob13,217,skoob13,Georgiy Tarasov,PostHog
7,MattPua,212,MattPua,Matt Pua,PostHog
8,jonmcwest,209,jonmcwest,Jon McCallum,@PostHog
9,tatoalo,204,tatoalo,Alessandro Pogliaghi,@posthog


## Reviews, labels, and dashboard-aligned views

In [7]:
display(read_df("SELECT * FROM reviews LIMIT 5"))

display(
    read_df(
        "SELECT label, COUNT(*) AS n FROM issue_labels GROUP BY 1 ORDER BY n DESC LIMIT 15"
    )
)

display(
    read_df('''
        SELECT reviewer_login, reviewed_others, distinct_authors
        FROM review_reach
        ORDER BY reviewed_others DESC
        LIMIT 15
    ''')
)

display(
    read_df(
        "SELECT label, COUNT(*) AS n FROM pr_labels GROUP BY 1 ORDER BY n DESC LIMIT 15"
    )
)


,reviewer_login,pr_number
0,pauldambra,84151
1,pauldambra,84106
2,pauldambra,83380
3,pauldambra,82409
4,pauldambra,82259


,label,n
0,enhancement,321
1,bug,219
2,team/product-analytics,80
3,team/feature-flags,75
4,feature/customer-analytics,70
5,team/data-warehouse,53
6,feature/feature-flags,52
7,feature/pipeline-sources,46
8,feature/desktop,43
9,feature/libraries,32


,reviewer_login,reviewed_others,distinct_authors
0,MarconLP,583,15
1,gantoine,489,77
2,Gilbert09,437,57
3,andrewm4894,344,41
4,rnegron,271,50
5,skoob13,267,44
6,rafaeelaudibert,253,47
7,webjunkie,210,43
8,tatoalo,182,32
9,pauldambra,179,43


,label,n
0,stamphog,4333
1,skip-agent-review,583
2,reviewhog,527
3,skip-inkeep-docs,365
4,feature/desktop,326
5,team/feature-flags,248
6,feature/feature-flags,166
7,run-ci-frontend,130
8,stale,118
9,automated,114


## PR title taxonomy

PostHog merged PR titles are almost all [Conventional Commits](https://www.conventionalcommits.org/): `type(scope): subject`.

In [8]:
COMMIT = re.compile(
    r"^(?P<type>\w+)(?:\((?P<scope>[^)]+)\))?\s*:\s*(?P<subject>.+)$",
    re.I,
)

def parse_pr_title(title: str) -> pd.Series:
    m = COMMIT.match((title or "").strip())
    if not m:
        return pd.Series({"commit_type": "other", "scope": None, "subject": title})
    return pd.Series(
        {
            "commit_type": m.group("type").lower(),
            "scope": (m.group("scope") or "").lower() or None,
            "subject": m.group("subject").strip(),
        }
    )

parsed = prs_df[["number", "title", "author_login"]].join(
    prs_df["title"].apply(parse_pr_title)
)
display(parsed["commit_type"].value_counts().to_frame("count"))
display(parsed["scope"].value_counts().head(15).to_frame("count"))


,count
commit_type,
fix,5580
feat,5322
chore,1991
refactor,257
perf,172
docs,48
ci,41
test,31
revert,25


,count
scope,
data-warehouse,1569
signals,456
ci,369
experiments,343
tasks,332
insights,321
replay-vision,299
llma,293
flags,245


In [9]:
display(
    parsed.groupby(["author_login", "commit_type"])
    .size()
    .unstack(fill_value=0)
    .sort_values("feat", ascending=False)
    .head(10)
)


commit_type,chore,ci,docs,feat,fix,other,perf,refactor,revert,test
author_login,,,,,,,,,,
Gilbert09,39,0,1,954,1210,0,1,6,1,2
andrewm4894,58,0,0,218,74,0,0,4,2,0
pauldambra,98,1,3,169,176,0,13,45,2,10
jonmcwest,9,0,1,151,46,0,0,2,0,0
sampennington,59,0,0,136,134,0,1,14,0,0
arthurdedeus,1,0,0,132,29,0,1,11,1,0
MattPua,28,0,2,118,55,0,1,7,1,0
skoob13,32,0,0,104,72,0,2,3,4,0
DanielVisca,12,0,1,89,37,0,1,0,0,0


## Issue title taxonomy

Issues are less structured: bracket tags, bug reports, feature requests, or freeform prose.

In [10]:
BRACKET = re.compile(r"^\[([^\]]+)\]", re.I)
BUG = re.compile(r"^bug report", re.I)

def classify_issue_title(title: str) -> pd.Series:
    t = (title or "").strip()
    low = t.lower()

    m = COMMIT.match(t)
    if m:
        return pd.Series({"issue_kind": "conventional", "tag": m.group("type").lower()})

    m = BRACKET.match(t)
    if m:
        return pd.Series({"issue_kind": "bracket", "tag": m.group(1).strip()})

    if BUG.match(low):
        return pd.Series({"issue_kind": "bug_report", "tag": "bug"})
    if "feature request" in low:
        return pd.Series({"issue_kind": "feature_request", "tag": "feature"})
    if "?" in t:
        return pd.Series({"issue_kind": "question", "tag": "question"})

    return pd.Series({"issue_kind": "freeform", "tag": None})

classified = issues_df.join(issues_df["title"].apply(classify_issue_title))
display(classified["issue_kind"].value_counts().to_frame("count"))


,count
issue_kind,
freeform,586
conventional,187
feature_request,120
bug_report,41
bracket,31
question,4


## Freeform issue themes (unsupervised)

TF-IDF + k-means on titles that did not match the rules above.

In [11]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

freeform = classified.loc[classified["issue_kind"] == "freeform", "title"].tolist()
X = TfidfVectorizer(max_features=500, stop_words="english").fit_transform(freeform)
labels = KMeans(n_clusters=8, random_state=0, n_init="auto").fit_predict(X)

for k in range(8):
    print(f"\n--- cluster {k} ---")
    for t in [freeform[i] for i, lab in enumerate(labels) if lab == k][:5]:
        print(f"  {t[:100]}")



--- cluster 0 ---
  ProseMirror-selectednode CSS class is added to all images in dashboards
  Renaming an action doesn't take effect until after a refresh.
  Nuxt error tracking docs use deprecated/incorrect `sourcemaps.project` key
  Sprint - May 4 to May 17
  Move `scheduled_at` out of workflow trigger

--- cluster 1 ---
  Handover notes for customer accounts
  Data migration to customer analytics
  Group analytics mega-issue
  Connect persons to accounts
  Add Device Make/Model for insights/web analytics

--- cluster 2 ---
  test
  Flaky test quarantined: tools/hogli-commands/hogli_commands/tests/test_test_runner.py
  Stabilize the multimodal trace Playwright test
  test
  SDK Compliance Test Harness Status

--- cluster 3 ---
  Add coupon data from Stripe to Stripe source
  GitHub warehouse source: deployment_statuses can never sync (PostHog GitHub App lacks deployments:re
  Repeat source map uploads fail with Symbol set already has a release ID
  Logs “Related errors” tab fails wi